In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **Step 1 — Check Kaggle Environment**

In [2]:
# Check Python version

import sys

print("Python version:", sys.version)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


# **Step 2 — Install the libraries**

In [3]:
!pip install -q transformers accelerate

# **Step 3 — Check Transformers**

In [4]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.0.0


# **Step 4 — Check GPU Availability**

In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cpu
GPU available: False


# **Step 5 — Load a Conversational Model**

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # float32 becuase CPU is using
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully on:", device)

Loading model: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on: cpu


# **Step 6 — Build the Chatbot Function**

In [7]:
# System prompt — you can change and check chatbot "personality" or can set business use-case
SYSTEM_PROMPT = "You are a helpful, friendly AI assistant for a business. Answer clearly and concisely."

# Conversation history here
conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def chat(user_message, max_new_tokens=200):
    """Ek user message le kar, model se reply generate karta hai aur history update karta hai."""
    conversation_history.append({"role": "user", "content": user_message})

    # Chat template apply and make prompt
    prompt = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # only new generated part
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

# **Step 7 — Test the Chatbot**

In [8]:
response = chat("Hello! Who are you and what can you help me with?")
print("Bot:", response)

Bot: I am an AI language model designed to assist users in providing information, answering questions, and generating text based on the prompts given to me. My purpose is to provide useful responses that are tailored to your specific needs. Please let me know how I can be of assistance to you today.


In [9]:
response = chat("Can you give me 3 tips for improving customer service in a small business?")
print("Bot:", response)

Bot: Certainly! Here are three tips to improve customer service in a small business:

1. **Personalization**: Make sure your customers feel valued by personalizing their interactions with you. This could include using their name when greeting them, offering personalized recommendations or solutions, and showing appreciation for their feedback.

2. **Satisfaction**: Keep your customers happy by being responsive and attentive. Regularly check in with your customers after they've placed an order or made a purchase to ensure satisfaction. Responding promptly to issues and resolving complaints efficiently can go a long way in maintaining customer loyalty.

3. **Training and Development**: Invest in training your team to handle customer inquiries effectively. Providing ongoing education about different customer interaction styles and how to communicate professionally can make a significant difference in how well your customers interact with your business. Additionally, regular evaluations an

# **Step 8 — Interactive Chat Loop**

In [10]:
print("Chatbot ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit"]:
        print("Bot: Goodbye!")
        break
    reply = chat(user_input)
    print("Bot:", reply)

Chatbot ready! Type 'exit' to stop.



You:  hello


Bot: Hello! How can I assist you today?


You:  can you answer in japanese?


Bot: はい、私に日本語で答えることができます。何かお手伝いできることがありますか？


...

Bot: بله، سأقوم بتقديم الرد باللغة العربية. كيف يمكنني مساعدتك اليوم؟


You:  i tell you lastly about urdu but you answer me in arabic.


Bot: بالطبع، يمكنك الإجابة على الأسئلة باللغة العربية. كيف يمكنني مساعدتك اليوم؟


You:  exit


Bot: Goodbye!


# **Next Steps — Making This Business-Ready**

# **Step 9 — Load a Bigger Model (Optional)**

In [11]:
# If you want better answers, you can switch to a bigger model.
# This works best if you have a GPU session enabled in Kaggle.
def load_model(model_name):
    # This function replaces the current model with a new one
    global tokenizer, model, device, MODEL_NAME

    MODEL_NAME = model_name
    print("Loading model:", MODEL_NAME)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print("Model loaded on:", device)

# Example: try a bigger model (needs GPU)
# load_model("Qwen/Qwen2.5-1.5B-Instruct")

# **Step 10 — Simple Knowledge Base Search (Basic RAG)**

In [12]:
# A small list of business facts (add your own here)
KNOWLEDGE_BASE = [
    "Our office hours are Monday to Saturday, 9 AM to 6 PM.",
    "Products can be returned within 7 days with a receipt.",
    "Delivery usually takes 3 to 5 working days.",
    "Customer support is available on WhatsApp and email.",
]

def find_relevant_facts(user_message):
    # Very simple search: check if any word from the fact
    # also appears in the user's message
    message_words = set(user_message.lower().split())
    matches = []

    for fact in KNOWLEDGE_BASE:
        fact_words = set(fact.lower().split())
        if message_words & fact_words:  # if there is any common word
            matches.append(fact)

    return matches

def chat_with_context(user_message, max_new_tokens=200):
    # Find any relevant facts first
    relevant_facts = find_relevant_facts(user_message)

    if relevant_facts:
        facts_text = "\n".join(f"- {f}" for f in relevant_facts)
        full_message = f"Company info:\n{facts_text}\n\nQuestion: {user_message}"
    else:
        full_message = user_message

    return chat(full_message, max_new_tokens=max_new_tokens)

In [13]:
response = chat_with_context("What are your office hours?")
print("Bot:", response)

Bot: Your office hours are from 9 AM to 6 PM on weekdays.
